# NEW MODEL

## Data loading and preprocessing

In [58]:
train_subjects = [2, 3, 10, 12, 13, 14, 16, 17, 18, 21]
val_subjects = [1, 5, 6, 7, 11, 15, 20]
test_subjects = [4, 8, 9, 19, 22] 

In [59]:
import numpy as np
import pandas as pd
def segment_data(df, subjects, w=2, o=0.75, fs=60):
    win_size = int(w * fs)
    step = int(win_size * (1 - o))
    
    # Order matters for the Multi-Branch CNN: 8 Ankle channels first, 4 Back channels last
    cols = [
        'ankleL_gyro_x', 'ankleL_gyro_y', 'ankleL_gyro_z',
        'ankleR_gyro_x', 'ankleR_gyro_y', 'ankleR_gyro_z',
        'back_gyro_x', 'back_gyro_y', 'back_gyro_z'
    ]
    
    X, y = [], []
    
    for sub in subjects:
        sub_df = df[df['subjectID'] == sub].reset_index(drop=True)
        data = sub_df[cols].values
        labels = sub_df['fog'].values
        
        for i in range(0, len(data) - win_size, step):
            window = data[i : i + win_size]
            # --- RELATIVE TRANSFORMATION ---
            # Subtract the first sample of the window from the whole window
            window_rel = window - window[0]
            
            X.append(window_rel)
            # Use the mode (most frequent) label for the window
            y.append(pd.Series(labels[i : i + win_size]).mode()[0])
            
    return np.array(X), np.array(y)

# Load your generated quaternions file
df_quat = pd.read_csv('sensor_data_complete.csv')

print("Segmenting datasets...")
X_train, y_train = segment_data(df_quat, train_subjects)
X_val, y_val     = segment_data(df_quat, val_subjects)
X_test, y_test   = segment_data(df_quat, test_subjects)

Segmenting datasets...


In [60]:
from sklearn.preprocessing import StandardScaler
import numpy as np

X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
# Supposons que X_train_gyro et X_test_gyro sont tes données (Batch, Channels, Time)
# La forme est (N, 9, 120)

# 1. On doit aplatir les données pour que le StandardScaler comprenne
# On passe de (N, 9, 120) à (N * 120, 9)
N_train = X_train.shape[0]
X_train_flat = X_train.transpose(0, 2, 1).reshape(-1, 9)

N_test = X_test.shape[0]
X_test_flat = X_test.transpose(0, 2, 1).reshape(-1, 9)

X_val_flat = X_val.transpose(0, 2, 1).reshape(-1, 9)
# 2. On initialise le scaler et on l'entraîne UNIQUEMENT sur le Train Set
scaler = StandardScaler()
X_train_scaled_flat = scaler.fit_transform(X_train_flat)
X_val_scaled_flat = scaler.transform(X_val_flat)

# On applique la même transformation au Test Set (sans le ré-entraîner !)
X_test_scaled_flat = scaler.transform(X_test_flat)

# 3. On redonne aux données leur forme 3D d'origine (Batch, Channels, Time)
X_train_gyro_scaled = X_train_scaled_flat.reshape(N_train, 120, 9).transpose(0, 2, 1)
X_test_gyro_scaled = X_test_scaled_flat.reshape(N_test, 120, 9).transpose(0, 2, 1)
X_val_gyro_scaled = X_val_scaled_flat.reshape(X_val.shape[0], 120, 9).transpose(0, 2, 1)

In [61]:
print("Shape Train:", X_train_gyro_scaled.shape)
print("Min Train:", np.min(X_train_gyro_scaled))
print("Max Train:", np.max(X_train_gyro_scaled))
print("Contient des NaN ?:", np.isnan(X_train_gyro_scaled).any())

Shape Train: (5236, 9, 120)
Min Train: -14.105890001804378
Max Train: 15.235960934734186
Contient des NaN ?: False


In [62]:
import torch
from torch.utils.data import Dataset, DataLoader

class FogDataset(Dataset):
    def __init__(self, X, y):
        # Convert to float32 and move to tensors
        # X shape from segmentation: (Samples, Time, Channels) -> e.g., (N, 120, 12)
        # PyTorch Conv1d expects: (Samples, Channels, Time) -> e.g., (N, 12, 120)
        self.X = torch.tensor(X, dtype=torch.float32).permute(0, 2, 1)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# --- Usage ---
# Assuming you already have X_train, y_train, etc., from your segmentation script
train_dataset = FogDataset(X_train_gyro_scaled, y_train)
val_dataset = FogDataset(X_val_gyro_scaled, y_val)
test_dataset = FogDataset(X_test_gyro_scaled, y_test)

# DataLoader handles batching and shuffling
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [63]:
from sklearn.utils.class_weight import compute_class_weight

# Calculate weights for the loss function
weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
# Convert to a tensor for PyTorch (pos_weight for BCEWithLogitsLoss)
pos_weight = torch.tensor([weights[1] / weights[0]], dtype=torch.float32).to(device)

print(f"Class weights: {weights} | Pos_weight: {pos_weight.item():.2f}")

Class weights: [0.62052619 2.57423795] | Pos_weight: 4.15


## Model architecture and training

In [64]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super(FocalLoss, self).__init__()
        # alpha > 0.5 favorise la classe minoritaire (FoG)
        self.alpha = alpha
        # gamma contrôle la concentration sur les cas difficiles
        self.gamma = gamma

    def forward(self, inputs, targets):
        # inputs doit être les "logits" bruts (sans Sigmoid dans le modèle)
        bce_loss = F.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss) # Probabilité que le modèle a assignée à la vraie classe
        focal_loss = self.alpha * (1 - pt)**self.gamma * bce_loss
        return focal_loss.mean()

In [65]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiBranchCNN(nn.Module):
    def __init__(self):
        super(MultiBranchCNN, self).__init__()
        
        # Branch 1: Ankles (8 channels: Left q0-q3 + Right q0-q3)
        self.ankle_branch = nn.Sequential(
            nn.Conv1d(6, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1) # Reduces to a single vector
        )
        
        # Branch 2: Back (4 channels: Back q0-q3)
        self.back_branch = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Dropout(0.5),
            nn.Conv1d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1)
        )
        
        # Fusion Layer: Combines 128 (Ankles) + 64 (Back) = 192 features
        self.classifier = nn.Sequential(
            nn.Linear(192, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # x is assumed to be (Batch, Channels, Time) -> (Batch, 12, 120)
        ankles = x[:, 0:6, :] # First 6 channels
        back = x[:, 6:9, :]  # Last 3 channels
        
        feat_ankles = self.ankle_branch(ankles).view(x.size(0), -1)
        feat_back = self.back_branch(back).view(x.size(0), -1)
        
        # Concatenate branches
        combined = torch.cat((feat_ankles, feat_back), dim=1)
        return self.classifier(combined)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = MultiBranchCNN().to(device)
print(f"Model initialized on: {device}")

Model initialized on: mps


In [66]:
import numpy as np

def train_model(model, train_loader, val_loader, epochs=50, lr=0.001):
    # Setup Optimizer and Loss
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3)
    #criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    criterion = nn.BCELoss()
    #criterion = FocalLoss(alpha=0.75, gamma=2.0)

    # Scheduler: Reduces LR if validation loss doesn't improve for 5 epochs
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
    
    best_val_loss = float('inf')
    
    for epoch in range(epochs):
        # --- TRAINING PHASE ---
        model.train()
        train_losses = []
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        
        # --- VALIDATION PHASE ---
        model.eval()
        val_losses = []
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs).squeeze()
                v_loss = criterion(outputs, labels)
                val_losses.append(v_loss.item())
        
        avg_train_loss = np.mean(train_losses)
        avg_val_loss = np.mean(val_losses)
        
        # Update Scheduler
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")
        
        # Save Best Model
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), 'best_multibranch_model_gyro_acc_2.pth')
            print("--> Model Saved!")

# --- START TRAINING ---
train_model(model, train_loader, val_loader)

Epoch 1/50 | Train Loss: 0.5116 | Val Loss: 0.5077 | LR: 0.001000
--> Model Saved!
Epoch 2/50 | Train Loss: 0.4866 | Val Loss: 0.4902 | LR: 0.001000
--> Model Saved!
Epoch 3/50 | Train Loss: 0.4788 | Val Loss: 0.4850 | LR: 0.001000
--> Model Saved!
Epoch 4/50 | Train Loss: 0.4734 | Val Loss: 0.4870 | LR: 0.001000
Epoch 5/50 | Train Loss: 0.4701 | Val Loss: 0.4807 | LR: 0.001000
--> Model Saved!
Epoch 6/50 | Train Loss: 0.4701 | Val Loss: 0.4832 | LR: 0.001000
Epoch 7/50 | Train Loss: 0.4671 | Val Loss: 0.4845 | LR: 0.001000
Epoch 8/50 | Train Loss: 0.4627 | Val Loss: 0.4792 | LR: 0.001000
--> Model Saved!
Epoch 9/50 | Train Loss: 0.4622 | Val Loss: 0.4769 | LR: 0.001000
--> Model Saved!
Epoch 10/50 | Train Loss: 0.4590 | Val Loss: 0.4762 | LR: 0.001000
--> Model Saved!
Epoch 11/50 | Train Loss: 0.4588 | Val Loss: 0.4809 | LR: 0.001000
Epoch 12/50 | Train Loss: 0.4515 | Val Loss: 0.5010 | LR: 0.001000
Epoch 13/50 | Train Loss: 0.4487 | Val Loss: 0.4957 | LR: 0.001000
Epoch 14/50 | Train

In [69]:
import torch
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

def evaluate_model(model, test_loader, device, threshold=0.7):
    model.eval()  # Set model to evaluation mode (disables Dropout/BatchNorm)
    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():  # No need to track gradients during testing
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            
            # Get model output
            outputs = model(inputs).squeeze()
            #raw_outputs = model(inputs).squeeze()
            #outputs = torch.sigmoid(raw_outputs).squeeze()  # Apply sigmoid to get probabilities
            # Store raw probabilities for AUC
            all_probs.extend(outputs.cpu().numpy())
            
            # Apply threshold to get binary classes
            preds = (outputs > threshold).float()
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_probs)

# --- EXECUTION ---
# Load the best weights saved during training
model.load_state_dict(torch.load('best_multibranch_model_gyro_acc_2.pth'))

y_true, y_pred, y_probs = evaluate_model(model, test_loader, device, threshold=0.4)

In [70]:
print("\n" + "="*30)
print("   FINAL TEST RESULTS")
print("="*30)

# 1. Classification Report
print(classification_report(y_true, y_pred, target_names=['Normal', 'FoG']))

# 2. AUC Score (measures how well the model separates the two classes)
auc = roc_auc_score(y_true, y_probs)
print(f"Test AUC-ROC: {auc:.4f}")

# 3. Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
print("\nConfusion Matrix:")
print(cm)


   FINAL TEST RESULTS
              precision    recall  f1-score   support

      Normal       0.81      0.97      0.88      1883
         FoG       0.39      0.08      0.14       474

    accuracy                           0.79      2357
   macro avg       0.60      0.52      0.51      2357
weighted avg       0.72      0.79      0.73      2357

Test AUC-ROC: 0.6445

Confusion Matrix:
[[1821   62]
 [ 435   39]]


In [71]:
from sklearn.metrics import precision_score, recall_score, f1_score
print("Threshold Optimization:")
print("Thresh | Precision | Recall | F1-Score")
print("-" * 40)

# Loop through thresholds from 0.3 to 0.8
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    # Apply the threshold to the raw probabilities
    y_pred_adjusted = (y_probs > t).astype(int)
    
    p = precision_score(y_true, y_pred_adjusted)
    r = recall_score(y_true, y_pred_adjusted)
    f = f1_score(y_true, y_pred_adjusted)
    
    print(f"  {t:.1f}  |   {p:.2f}    |  {r:.2f}  |   {f:.2f}")

Threshold Optimization:
Thresh | Precision | Recall | F1-Score
----------------------------------------
  0.3  |   0.30    |  0.41  |   0.34
  0.4  |   0.39    |  0.08  |   0.14
  0.5  |   0.00    |  0.00  |   0.00
  0.6  |   0.00    |  0.00  |   0.00
  0.7  |   0.00    |  0.00  |   0.00
  0.8  |   0.00    |  0.00  |   0.00


/Users/zoemonnard/Documents/FoG-STAR/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zoemonnard/Documents/FoG-STAR/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/zoemonnard/Documents/FoG-STAR/venv/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capi

In [55]:
num_normal = (y_train == 0).sum()
num_fog = (y_train == 1).sum()

# Calculate the weight: How much more we should care about FoG
weight_ratio = num_normal / num_fog 
print(f"FoG Weight Penalty: {weight_ratio:.2f}")
pos_weight = torch.tensor([weight_ratio], dtype=torch.float32).to(device)

FoG Weight Penalty: 3.12
